In [7]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
model= ChatOpenAI(model_name="gpt-4o-mini")

In [4]:
class EvaluationSchema(BaseModel):
    score: int = Field(description="Score out of 10.", ge=0, le=10)
    feedback: str = Field( description="Detailed feedback for the Essay in 100 words or less")

In [5]:
structured_model = model.with_structured_output(EvaluationSchema)

In [8]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [12]:
def evaluate_language(state: UPSCState):
    prompt = f"Evaluate the following essay for language quality and provide a score out of 10 and feedback in 100 words or less:\n\n{state['essay']}"
    result = structured_model.invoke(prompt)
    return {"language_feedback": result.feedback, "individual_scores": [result.score]}

In [13]:
def evaluate_analysis(state: UPSCState):
    prompt = f"Evaluate the following essay for depth of analysis quality and provide a score out of 10 and feedback in 100 words or less:\n\n{state['essay']}"
    result = structured_model.invoke(prompt)
    return {"analysis_feedback": result.feedback, "individual_scores": [result.score]}

In [14]:
def evaluate_thought(state: UPSCState):
    prompt = f"Evaluate the following essay for clarity of thought and provide a score out of 10 and feedback in 100 words or less:\n\n{state['essay']}"
    result = structured_model.invoke(prompt)
    return {"clarity_feedback": result.feedback, "individual_scores": [result.score]}

In [18]:
def final_evaluation(state: UPSCState): 
    #feedback for overall evaluation
    prompt = f"Based on the following feedbacks, provide an overall evaluation of the essay in 100 words or less:\n\nLanguage-quality Feedback: {state['language_feedback']}\n depth of Analysis Feedback: {state['analysis_feedback']}\nClarity of thought Feedback: {state['clarity_feedback']}"
    overall_feedback = model.invoke(prompt).content

    #avg_score calculation
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {"overall_feedback": overall_feedback, "avg_score": avg_score}

In [21]:
graph = StateGraph(UPSCState)

#add nodes to the graph
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

#add edges to the graph
graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation',END)

workflow = graph.compile()

In [23]:
essay="""
To observe Indian civic sense is to stand squarely on the threshold of an Indian home. Step inside, and you will almost invariably find immaculate tiled floors, shoes neatly removed at the entrance, and brass fixtures polished to a warm shine. Step outside onto the street, however, and the scenery abruptly transforms: empty wrappers drifting along the curb, paan stains blooming across boundary walls, and a chorus of impatient horns honking at a red light that still has fifteen seconds to run. As an amateur observer looking closely at this everyday landscape, one quickly realizes that India's civic consciousness is not absent; it is curiously, stubbornly fenced in.

The heart of the issue lies in where we draw the boundary between the sacred private sanctuary and the orphaned public square. Culturally, the private domain demands fierce devotion and meticulous care. The street, the park, the suburban train compartment, and the pavement, by contrast, are treated as no-man's-land. When a plastic bottle is tossed out of a car window, it is rarely an act of open defiance. Instead, it springs from a quiet, subconscious assumption that once an object crosses the private threshold, it ceases to be one's responsibility—instantly becoming the problem of the municipality or someone further down the social ladder. Deep-seated historical hierarchies around sanitation have long insulated many from feeling personal ownership over public hygiene.

Beyond cleanliness, civic sense is written into the social contracts of shared space and movement. Standing in an Indian queue is often an exercise in defensive posture: leave an arm’s length of breathing room, and someone will inevitably slide into the gap. On the roads, painted lane markings are treated as mild decorative suggestions, and the pedestrian zebra crossing is virtually invisible. Much of this daily friction traces back to an ingrained psychology of scarcity. In an environment where seats, tickets, hospital beds, and open asphalt have historically required elbowing through a crowd, patience is easily mistaken for surrender, and courtesy can feel like losing an edge.

Yet dismissing the Indian civic instinct as entirely broken overlooks its remarkable, latent warmth. The same commuter who cuts a traffic lane without blinking will step out into pouring rain to help a total stranger push a stalled auto-rickshaw out of a flooded ditch. During civic crises—from seasonal floods to community emergencies—informal neighborhood networks mobilize food, shelter, and rescue efforts with astonishing speed. Indian civic sense is deeply relational rather than institutional. It responds vividly to human faces in distress, even while remaining indifferent to the concrete infrastructure around them.

Closing that divide is the real challenge of India’s ongoing urbanization. A gradual shift is already visible among younger generations, visible in community waste-segregation initiatives, cyclist groups advocating for road etiquette, and a broader public conversation that treats cleanliness as a shared duty rather than someone else's job. Genuine civic maturity will not come merely from stiffer traffic fines or more municipal sweepers. It will take root when we collectively realize that the public street outside our front gate is not an alien void, but a direct extension of the home we take so much pride in keeping clean."""


In [24]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)

{'essay': "\nTo observe Indian civic sense is to stand squarely on the threshold of an Indian home. Step inside, and you will almost invariably find immaculate tiled floors, shoes neatly removed at the entrance, and brass fixtures polished to a warm shine. Step outside onto the street, however, and the scenery abruptly transforms: empty wrappers drifting along the curb, paan stains blooming across boundary walls, and a chorus of impatient horns honking at a red light that still has fifteen seconds to run. As an amateur observer looking closely at this everyday landscape, one quickly realizes that India's civic consciousness is not absent; it is curiously, stubbornly fenced in.\n\nThe heart of the issue lies in where we draw the boundary between the sacred private sanctuary and the orphaned public square. Culturally, the private domain demands fierce devotion and meticulous care. The street, the park, the suburban train compartment, and the pavement, by contrast, are treated as no-man's